# Cat Behaviour Video Classifier

Create, train and export a cat behaviour classifier from labelled track embeddings.

## Configure notebook

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
import base64
import copy
import json
import random
from collections import Counter, defaultdict
from dataclasses import InitVar, dataclass, field
from enum import IntEnum
from itertools import combinations
from tempfile import TemporaryDirectory
from time import perf_counter_ns

import cv2
import ipywidgets as widgets
import joblib
import numpy as np
import onnx
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from matplotlib import pyplot as plt
from onnxruntime.quantization import QuantType, quantize_dynamic
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    ConfusionMatrixDisplay,
    average_precision_score,
    precision_recall_curve,
)
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset

from tracking import embedding_model_sha256, behaviour_session, classify_behaviour
from utils import get_best_device, get_video_hashes

SEED = 42

In [ ]:
# Define constants
PANEL_WIDTH = 480
WINDOW_POWER = 5
WINDOW_SIZE = 2**WINDOW_POWER
DILATIONS = [2**exponent for exponent in range(WINDOW_POWER)]
BATCH_SIZE = 64
HIDDEN_CHANNELS = 64
KERNEL_SIZE = 2
DROPOUT = 0.1
MAX_EPOCHS = 50
PATIENCE = 8
VALIDATION_FRACTION = 0.5
DEVICE = get_best_device()

In [ ]:
# Define paths
DATA_ROOT = PROJECT_ROOT / "datasets" / "behaviour_dataset"
ANN_PATH = DATA_ROOT / "behaviour_labels.json"
TRACK_FILES = sorted(DATA_ROOT.glob("track-*.joblib"))
TCN_CHECKPOINT_PATH = PROJECT_ROOT / "models_staging" / "behaviour_tcn.pt"
GRU_CHECKPOINT_PATH = PROJECT_ROOT / "models_staging" / "behaviour_gru.pt"

## Load and validate tracks

In [ ]:
class AnnotationStatus(IntEnum):
    COMPLETE = 0
    VIDEO_UNLABELLED = 1
    VIDEO_UNLABELLED_CAT = 2
    PARTIAL = 3
    OVERLAPPING = 4
    DUPLICATED = 5


@dataclass
class TrackInfo:
    video_path: str
    cat_id: str
    frame_hashes: list[str | None]
    images: dict[str, np.ndarray]
    embeddings: dict[str, np.ndarray]
    embedding_model_sha256: str
    track_path: Path
    annotations: InitVar[list[dict]]
    video_name: str = field(init=False)
    annotation_status: AnnotationStatus = field(init=False)
    annotation_start_map: dict[str, str] | None = field(init=False, default=None)
    annotation_frame_offset_map: dict[str, int] | None = field(init=False, default=None)
    behaviour_map: dict[str, str] | None = field(init=False, default=None)

    def __post_init__(self, annotations: list[dict]) -> None:
        expected_hash = embedding_model_sha256()
        if self.embedding_model_sha256 != expected_hash:
            raise ValueError(
                f"Embedding model hash mismatch for {self.video_name} - {self.cat_id}"
            )
        self.video_name = Path(self.video_path).name
        self.annotation_status = self._join_annotations(annotations)

    def __len__(self) -> int:
        return len(self.frame_hashes)

    def _join_annotations(self, annotations: list[dict]) -> AnnotationStatus:
        # check if the video is unlabelled
        video_annotations = [
            a for a in annotations if a["video_name"] == self.video_name
        ]
        if not video_annotations:
            return AnnotationStatus.VIDEO_UNLABELLED

        # check for cat specific annotations
        cat_video_annotations = [
            ann for ann in video_annotations if ann["cat_id"] == self.cat_id
        ]
        if not cat_video_annotations:
            return AnnotationStatus.VIDEO_UNLABELLED_CAT

        # create hash map
        hash_map = {h: i for i, h in enumerate(get_video_hashes(self.video_path))}

        # convert annotations to index based windows
        windows = []
        for ann in cat_video_annotations:
            start = hash_map[ann["frame_hash_start"]]
            end = hash_map[ann["frame_hash_end"]]
            windows.append((start, end, ann["behaviour"], ann["frame_hash_start"]))

        # create behaviour map for each detected track frame
        self.behaviour_map = {}
        self.annotation_start_map = {}
        self.annotation_frame_offset_map = {}
        for frame_hash in [h for h in self.frame_hashes if h]:
            matches = [
                (b, h, s) for s, e, b, h in windows if s <= hash_map[frame_hash] <= e
            ]
            if len(matches) > 1:
                return AnnotationStatus.OVERLAPPING
            if not matches:
                return AnnotationStatus.PARTIAL
            behaviour, start_hash, start = matches[0]
            self.behaviour_map[frame_hash] = behaviour
            self.annotation_start_map[frame_hash] = start_hash
            self.annotation_frame_offset_map[frame_hash] = hash_map[frame_hash] - start

        return AnnotationStatus.COMPLETE


# load annotations and track information
annotations = json.loads(ANN_PATH.read_text()) if ANN_PATH.exists() else []
all_tracks = [
    TrackInfo(**joblib.load(p), track_path=p, annotations=annotations)
    for p in TRACK_FILES
]

In [ ]:
# mark overlapping tracks as duplicates
tracks_by_key = defaultdict(list)
for track in all_tracks:
    tracks_by_key[(track.video_name, track.cat_id)].append(track)
duplicated_tracks = []
for (video_name, cat_id), tracks in tracks_by_key.items():
    for t1, t2 in combinations(tracks, 2):
        shared_frames = (set(t1.frame_hashes) & set(t2.frame_hashes)) - {None}
        if shared_frames:
            t1.annotation_status = AnnotationStatus.DUPLICATED
            t2.annotation_status = AnnotationStatus.DUPLICATED

In [ ]:
# categorise tracks based status
complete_tracks = [
    t for t in all_tracks if t.annotation_status == AnnotationStatus.COMPLETE
]
incorrectly_labelled_tracks = [
    t
    for t in all_tracks
    if t.annotation_status
    in [
        AnnotationStatus.VIDEO_UNLABELLED_CAT,
        AnnotationStatus.PARTIAL,
        AnnotationStatus.OVERLAPPING,
        AnnotationStatus.DUPLICATED,
    ]
]
unprocessed_videos = {a["video_name"] for a in annotations} - {
    t.video_name for t in complete_tracks
}
unlabelled_videos = {
    t.video_name
    for t in all_tracks
    if t.annotation_status == AnnotationStatus.VIDEO_UNLABELLED
}
print(f"Complete tracks: {len(complete_tracks)}")
print(f"Incorrectly labelled tracks: {len(incorrectly_labelled_tracks)}")
print(f"Unlabelled videos: {len(unlabelled_videos)}")
for video_name in sorted(unlabelled_videos):
    print(f"  {video_name}")
print(f"Unprocessed videos: {len(unprocessed_videos)}")
for video_name in sorted(unprocessed_videos):
    print(f"  {video_name}")

## Inspect invalid tracks

In [ ]:
current_video = {"path": None, "images": [], "hashes": [], "height": None}


def image_to_html(image: np.ndarray, height: int | None = None) -> str:
    success, encoded = cv2.imencode(".jpg", image)
    if not success:
        raise RuntimeError("OpenCV could not encode the preview frame")
    data = base64.b64encode(encoded.tobytes()).decode("ascii")
    size = (
        f"width:100%;height:{height}px;object-fit:contain"
        if height
        else "width:100%;height:auto"
    )
    return f'<img src="data:image/jpeg;base64,{data}" style="display:block;{size}">'


def load_video(track: TrackInfo) -> None:
    if current_video["path"] == track.video_path:
        return
    capture = cv2.VideoCapture(track.video_path)
    images = []
    while True:
        success, image = capture.read()
        if not success:
            break
        images.append(image)
    capture.release()
    hashes = list(get_video_hashes(track.video_path))
    height = None
    if images:
        h, w = images[0].shape[:2]
        height = round(PANEL_WIDTH * h / w)
    current_video.update(
        path=track.video_path, images=images, hashes=hashes, height=height
    )


def make_panel() -> tuple[widgets.HTML, widgets.IntSlider, widgets.HTML]:
    view = widgets.HTML(
        layout=widgets.Layout(width=f"{PANEL_WIDTH}px", border="1px solid #555")
    )
    slider = widgets.IntSlider(
        description="Index",
        min=0,
        max=0,
        value=0,
        continuous_update=True,
        layout=widgets.Layout(width=f"{PANEL_WIDTH}px"),
    )
    return view, slider, widgets.HTML()


track_dropdown = widgets.Dropdown(
    options=[
        (f"{t.video_name} | {t.cat_id} | {t.annotation_status.name}", i)
        for i, t in enumerate(incorrectly_labelled_tracks)
    ],
    description="Track",
    layout=widgets.Layout(width=f"{PANEL_WIDTH * 2}px"),
)
video_view, video_slider, video_status = make_panel()
track_view, track_slider, track_status = make_panel()


def refresh_video(*_) -> None:
    images, hashes = current_video["images"], current_video["hashes"]
    if not images:
        video_view.value = ""
        video_status.value = "<b>Raw video</b> &nbsp; no decodable frames"
        return
    index = video_slider.value
    frame_hash = hashes[index] if index < len(hashes) else "-"
    video_status.value = (
        f"<b>Raw video</b> &nbsp; index {index}/{len(images) - 1}"
        f" &nbsp; hash: {frame_hash}"
    )
    video_view.value = image_to_html(images[index])


def refresh_track(*_) -> None:
    track = incorrectly_labelled_tracks[track_dropdown.value]
    index = track_slider.value
    frame_hash = track.frame_hashes[index]
    track_status.value = (
        f"<b>Track</b> &nbsp; index {index}/{len(track) - 1}"
        f" &nbsp; hash: {frame_hash}"
    )
    image = track.images.get(frame_hash)
    height = current_video["height"]
    if image is not None:
        track_view.value = image_to_html(image, height=height)
    else:
        track_view.value = f'<div style="height:{height or 0}px"></div>'


def on_track_change(*_) -> None:
    track = incorrectly_labelled_tracks[track_dropdown.value]
    load_video(track)
    video_slider.value = 0
    video_slider.max = max(len(current_video["images"]) - 1, 0)
    track_slider.value = 0
    track_slider.max = max(len(track) - 1, 0)
    refresh_video()
    refresh_track()


track_dropdown.observe(on_track_change, names="value")
video_slider.observe(refresh_video, names="value")
track_slider.observe(refresh_track, names="value")

In [ ]:
if incorrectly_labelled_tracks:
    display(
        widgets.VBox(
            [
                track_dropdown,
                widgets.HBox(
                    [
                        widgets.VBox([video_slider, video_status, video_view]),
                        widgets.VBox([track_slider, track_status, track_view]),
                    ]
                ),
            ]
        )
    )
    on_track_change()
else:
    print("No partial or invalid tracks to inspect.")

## Display class imbalance

In [ ]:
# calculate class imbalance
track_class_counts, frame_class_counts = Counter(), Counter()
for t in complete_tracks:
    track_class_counts.update((t.cat_id, b) for b in set(t.behaviour_map.values()))
    frame_class_counts.update((t.cat_id, b) for b in t.behaviour_map.values())
track_class_counts = pd.Series(track_class_counts).rename_axis().unstack()
frame_class_counts = pd.Series(frame_class_counts).rename_axis().unstack()

In [ ]:
# display class imbalance
fig, ax_all = plt.subplots(ncols=2, nrows=2, sharex=True, sharey="row", figsize=(15, 8))
for ax_row, counts, name in zip(
    ax_all, [track_class_counts, frame_class_counts], ["Track", "Frame"]
):
    counts.columns = [c.replace(" ", "\n") for c in counts.columns]
    for ax, (cat, ser) in zip(ax_row, counts.iterrows()):
        ser.plot(kind="bar", ax=ax, title=cat)
        ax.set_ylabel(f"{name} count")
        ax.tick_params(axis="x", labelrotation=0)
fig.tight_layout()

## Create model dataset

In [ ]:
BEHAVIOURS = sorted({b for t in complete_tracks for b in t.behaviour_map.values()})
BEHAVIOUR_TO_ID = {b: i for i, b in enumerate(BEHAVIOURS)}


class BehaviourWindowDataset(Dataset):
    """Causal windows ending on each detected frame; embeddings pulled on access"""

    def __init__(
        self,
        tracks: list[TrackInfo],
        window_size: int,
        class_counts: Counter | None = None,
    ) -> None:
        self.tracks = tracks
        self.window_size = window_size

        # forward fill missed detections and index samples on detected frames
        self.ffill_hashes = []
        self.samples, self.keys = [], []
        self.track_names = []
        self.annotation_start_hashes = []
        self.annotation_frame_offsets = []
        for track_idx, track in enumerate(tracks):
            frame_hashes_ffill, last_hash = [], None
            for pos, frame_hash in enumerate(track.frame_hashes):
                if frame_hash is not None:
                    last_hash = frame_hash
                    self.samples.append((track_idx, pos))
                    self.keys.append((track.cat_id, track.behaviour_map[frame_hash]))
                    self.track_names.append(track.track_path.name)
                    self.annotation_start_hashes.append(
                        track.annotation_start_map[frame_hash]
                    )
                    self.annotation_frame_offsets.append(
                        track.annotation_frame_offset_map[frame_hash]
                    )
                frame_hashes_ffill.append(last_hash)
            self.ffill_hashes.append(frame_hashes_ffill)
        self.targets = [BEHAVIOUR_TO_ID[b] for _, b in self.keys]

        # compute class weights for balanced sampling
        self.class_counts = class_counts if class_counts else Counter(self.keys)
        weights = np.array([1.0 / max(self.class_counts[k], 1) for k in self.keys])
        self.weights = weights / weights.mean()

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(
        self, idx: int
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, dict[str, str | int]]:
        track_idx, pos = self.samples[idx]
        track, hashes = self.tracks[track_idx], self.ffill_hashes[track_idx]
        positions = np.clip(np.arange(pos - self.window_size + 1, pos + 1), 0, None)
        x = np.stack([track.embeddings[hashes[p]] for p in positions], axis=1)
        return (
            torch.as_tensor(x, dtype=torch.float32),
            torch.tensor(self.targets[idx]),
            torch.tensor(self.weights[idx]),
            {
                "annotation_start_hash": self.annotation_start_hashes[idx],
                "annotation_frame_offset": self.annotation_frame_offsets[idx],
            },
        )

In [ ]:
# train - validation splits by video
groups = [t.video_name for t in complete_tracks]
train_idx, val_idx = next(
    GroupShuffleSplit(
        n_splits=1, test_size=VALIDATION_FRACTION, random_state=SEED
    ).split(complete_tracks, groups=groups)
)
train_ds = BehaviourWindowDataset([complete_tracks[i] for i in train_idx], WINDOW_SIZE)
val_ds = BehaviourWindowDataset(
    [complete_tracks[i] for i in val_idx],
    WINDOW_SIZE,
    class_counts=train_ds.class_counts,
)
print(f"Train: {len(train_ds)} windows; validation: {len(val_ds)} windows")

# Create model

In [ ]:
class CausalConv1d(torch.nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super().__init__()
        self.left_padding = (kernel_size - 1) * dilation
        self.conv = torch.nn.Conv1d(
            in_channels, out_channels, kernel_size, dilation=dilation
        )

    def forward(self, inputs):
        return self.conv(F.pad(inputs, (self.left_padding, 0)))


class TemporalResidualBlock(torch.nn.Module):
    def __init__(self, in_channels, channels, kernel_size, dilation, dropout):
        super().__init__()
        self.layers = torch.nn.Sequential(
            CausalConv1d(in_channels, channels, kernel_size, dilation),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            CausalConv1d(channels, channels, kernel_size, dilation),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
        )
        self.residual = (
            torch.nn.Identity()
            if in_channels == channels
            else torch.nn.Conv1d(in_channels, channels, 1)
        )

    def forward(self, inputs):
        return F.relu(self.layers(inputs) + self.residual(inputs))


class TemporalCausalClassifier(torch.nn.Module):
    def __init__(
        self,
        input_channels,
        num_classes,
        hidden_channels=HIDDEN_CHANNELS,
        kernel_size=KERNEL_SIZE,
        dilations=DILATIONS,
        dropout=DROPOUT,
    ):
        super().__init__()
        self.receptive_field = 1 + 2 * (kernel_size - 1) * sum(dilations)
        self.blocks = torch.nn.Sequential(
            *[
                TemporalResidualBlock(
                    input_channels if index == 0 else hidden_channels,
                    hidden_channels,
                    kernel_size,
                    dilation,
                    dropout,
                )
                for index, dilation in enumerate(dilations)
            ]
        )
        self.classifier = torch.nn.Linear(hidden_channels, num_classes)

    def forward_features(self, inputs):
        return self.blocks(inputs)

    def forward(self, inputs):
        return self.classifier(self.forward_features(inputs)[:, :, -1])


class GRUFrameClassifier(torch.nn.Module):
    def __init__(
        self,
        input_channels,
        num_classes,
        hidden_channels=HIDDEN_CHANNELS,
        dropout=DROPOUT,
    ):
        super().__init__()
        self.cell = torch.nn.GRUCell(input_channels, hidden_channels)
        self.dropout = torch.nn.Dropout(dropout)
        self.classifier = torch.nn.Linear(hidden_channels, num_classes)

    def update_state(self, embedding, hidden_state=None):
        return self.cell(embedding, hidden_state)

    def classify_state(self, hidden_state):
        return self.classifier(self.dropout(hidden_state))

    def forward(self, embedding, hidden_state=None):
        next_state = self.update_state(embedding, hidden_state)
        return self.classify_state(next_state), next_state


class GRUSequenceClassifier(torch.nn.Module):
    def __init__(
        self,
        input_channels,
        num_classes,
        hidden_channels=HIDDEN_CHANNELS,
        dropout=DROPOUT,
    ):
        super().__init__()
        self.frame_model = GRUFrameClassifier(
            input_channels, num_classes, hidden_channels, dropout
        )

    def forward(self, inputs):
        hidden_state = None
        for embedding in inputs.unbind(dim=2):
            hidden_state = self.frame_model.update_state(embedding, hidden_state)
        return self.frame_model.classify_state(hidden_state)

In [ ]:
def aggregate_frames(frames):
    correct = frames["prediction"] == frames["target"]
    if correct.any():
        latency = frames.loc[correct, "annotation_frame_offset"].min()
        stability = correct.loc[frames["annotation_frame_offset"] >= latency].mean()
    else:
        latency, stability = np.inf, 0.0
    return pd.Series(
        {
            "target": frames["target"].iloc[0],
            "prediction": frames["prediction"].mode().iloc[0],
            "latency_frames": latency,
            "stability": stability,
        }
    )

In [ ]:
def run_epoch(model, loader, optimizer=None):
    # configure train or evaluation mode and collect epoch totals
    is_training = optimizer is not None
    model.train(is_training)
    total_loss, total_weight = 0.0, 0.0
    targets_all, predictions_all, probabilities_all = [], [], []
    metadata_all = {"annotation_start_hash": [], "annotation_frame_offset": []}
    with torch.set_grad_enabled(is_training):
        for inputs, targets, weights, metadata in loader:
            # compute loss
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            weights = weights.to(device=DEVICE, dtype=inputs.dtype)
            logits = model(inputs)
            loss_sum = (
                F.cross_entropy(logits, targets, reduction="none") * weights
            ).sum()

            # update model parameters only during training
            if is_training:
                optimizer.zero_grad(set_to_none=True)
                (loss_sum / weights.sum()).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            # accumulate weighted loss and predictions across batches
            total_loss += loss_sum.item()
            total_weight += weights.sum().item()
            targets_all.extend(targets.cpu().tolist())
            predictions_all.extend(logits.detach().argmax(dim=1).cpu().tolist())
            probabilities_all.extend(logits.detach().softmax(dim=1).cpu().tolist())
            metadata_all["annotation_start_hash"].extend(
                metadata["annotation_start_hash"]
            )
            metadata_all["annotation_frame_offset"].extend(
                metadata["annotation_frame_offset"].tolist()
            )

    # report weighted loss and unweighted classification metrics
    return (
        {
            "loss": total_loss / total_weight,
            "accuracy": accuracy_score(targets_all, predictions_all),
            "macro_f1": f1_score(
                targets_all,
                predictions_all,
                labels=range(len(BEHAVIOURS)),
                average="macro",
                zero_division=0,
            ),
        },
        targets_all,
        predictions_all,
        np.asarray(probabilities_all),
        metadata_all,
    )


def format_metrics(metrics):
    return ", ".join(f"{key}: {value:.3f}" for key, value in metrics.items())


def fit_model(model, train_ds, val_ds, seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    generator = torch.Generator().manual_seed(seed)

    # create optimizer and data loaders for the supplied model
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, generator=generator)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    best_state, best_score, best_epoch = None, (-1.0, -float("inf")), 0
    for epoch in range(1, MAX_EPOCHS + 1):
        # train and validate once per epoch and print both sets of metrics
        train_metrics, _, _, _, _ = run_epoch(model, train_loader, optimizer)
        metrics, _, _, _, _ = run_epoch(model, val_loader)
        print(
            f"epoch {epoch:02d} | train: {format_metrics(train_metrics)} | val: {format_metrics(metrics)}",
            flush=True,
        )

        # retain the best validation macro-F1, using loss to break ties
        score = (metrics["macro_f1"], -metrics["loss"])
        if score > best_score:
            best_score, best_epoch = score, epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }

        # stop when validation has not improved for the patience period
        if epoch - best_epoch >= PATIENCE:
            break

    # restore the best weights and return their validation results
    model.load_state_dict(best_state)
    metrics, targets, predictions, probabilities, metadata = run_epoch(
        model, val_loader
    )
    print(f"{epoch} epochs, best {best_epoch}, {format_metrics(metrics)}")
    prediction_details_frame = pd.DataFrame(
        {
            "target": targets,
            "prediction": predictions,
            "track_name": val_ds.track_names,
            "annotation_start_hash": metadata["annotation_start_hash"],
            "annotation_frame_offset": metadata["annotation_frame_offset"],
            "probability": probabilities.max(axis=1),
        }
    )
    for class_id, behaviour in enumerate(BEHAVIOURS):
        prediction_details_frame[f"probability_{behaviour}"] = probabilities[
            :, class_id
        ]
    return {
        "best_epoch": best_epoch,
        "epochs_run": epoch,
        "metrics": metrics,
        "prediction_details_frame": prediction_details_frame,
        "prediction_details_event": prediction_details_frame.groupby(
            ["track_name", "annotation_start_hash"]
        )
        .apply(aggregate_frames)
        .reset_index(),
    }

In [ ]:
# Create and train models
tcn_config = {
    "input_channels": train_ds[0][0].shape[0],
    "num_classes": len(BEHAVIOURS),
    "hidden_channels": HIDDEN_CHANNELS,
    "kernel_size": KERNEL_SIZE,
    "dilations": DILATIONS,
    "dropout": DROPOUT,
}
gru_config = {
    "input_channels": tcn_config["input_channels"],
    "num_classes": tcn_config["num_classes"],
    "hidden_channels": HIDDEN_CHANNELS,
    "dropout": DROPOUT,
}
model_specs = {
    "TCN": (TemporalCausalClassifier, tcn_config),
    "GRU": (GRUSequenceClassifier, gru_config),
}
models, results = {}, {}
for model_name, (model_class, config) in model_specs.items():
    torch.manual_seed(SEED)
    models[model_name] = model_class(**config).to(DEVICE)
    results[model_name] = fit_model(models[model_name], train_ds, val_ds, seed=SEED)

In [ ]:
# Save a checkpoint for each trained model
checkpoint_configs = {"TCN": tcn_config, "GRU": gru_config}
checkpoint_paths = {
    "TCN": TCN_CHECKPOINT_PATH,
    "GRU": GRU_CHECKPOINT_PATH,
}
for model_name, trained_model in models.items():
    checkpoint = {
        "state_dict": {
            key: value.detach().cpu()
            for key, value in trained_model.state_dict().items()
        },
        "model_config": checkpoint_configs[model_name],
        "behaviours": BEHAVIOURS,
        "window_size": WINDOW_SIZE,
        "embedding_model_sha256": embedding_model_sha256(),
    }
    checkpoint_path = checkpoint_paths[model_name]
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(checkpoint, checkpoint_path)

## Evaluate behaviour classifiers

In [ ]:
def summarise_events(event_details):
    event_details = event_details.copy()
    event_details["target"] = event_details["target"].map(dict(enumerate(BEHAVIOURS)))
    summary = event_details.groupby("target").agg(
        event_count=("latency_frames", "size"),
        latency_frames=("latency_frames", "median"),
        stability=("stability", "median"),
    )
    summary.index.name = None
    summary[["latency_frames", "stability"]] = summary[
        ["latency_frames", "stability"]
    ].round(2)
    return summary


def plot_evaluation(frame_details, event_details, model_name):
    # initialise figure
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), layout="constrained")

    # confusion matrices
    for axis, details, level in zip(
        axes[:2], [frame_details, event_details], ["Frame", "Event"]
    ):
        ConfusionMatrixDisplay.from_predictions(
            details["target"],
            details["prediction"],
            labels=list(range(len(BEHAVIOURS))),
            display_labels=BEHAVIOURS,
            xticks_rotation=90,
            cmap="Blues",
            colorbar=False,
            ax=axis,
        )
        axis.set_title(f"{level} confusion matrix")

    # pr curve
    for class_id, behaviour in enumerate(BEHAVIOURS):
        binary_targets = frame_details["target"] == class_id
        probabilities = frame_details[f"probability_{behaviour}"]
        precision, recall, _ = precision_recall_curve(binary_targets, probabilities)
        average_precision = average_precision_score(binary_targets, probabilities)
        label = f"{behaviour}: AP={average_precision:.2f}"
        axes[2].step(recall, precision, where="post", label=label)
    axes[2].set(title="Frame PR curve", xlabel="Recall", ylabel="Precision")
    axes[2].legend(loc="lower left")
    fig.suptitle(f"{model_name}: validation diagnostics")
    plt.show()

In [ ]:
# Evaluate and summarise model results for each behaviour model
for model_name, result in results.items():
    print(f"--- {model_name.upper()} validation ---")
    print(
        classification_report(
            result["prediction_details_frame"]["target"],
            result["prediction_details_frame"]["prediction"],
            labels=list(range(len(BEHAVIOURS))),
            target_names=BEHAVIOURS,
            zero_division=0,
        )
    )
    print(summarise_events(result["prediction_details_event"]))
    plot_evaluation(
        result["prediction_details_frame"],
        result["prediction_details_event"],
        model_name,
    )

## Export the model to ONNX

In [ ]:
class GRUFrameExport(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        weights = {
            "input_weight": model.cell.weight_ih.T,
            "recurrent_weight": model.cell.weight_hh.T,
            "input_bias": model.cell.bias_ih,
            "recurrent_bias": model.cell.bias_hh,
            "output_weight": model.classifier.weight.T,
            "output_bias": model.classifier.bias,
        }
        for name, tensor in weights.items():
            self.register_buffer(name, tensor.detach().contiguous())

    def forward(self, embedding, hidden_state):
        input_reset, input_update, input_new = (
            embedding @ self.input_weight + self.input_bias
        ).chunk(3, dim=1)
        state_reset, state_update, state_new = (
            hidden_state @ self.recurrent_weight + self.recurrent_bias
        ).chunk(3, dim=1)
        reset = torch.sigmoid(input_reset + state_reset)
        update = torch.sigmoid(input_update + state_update)
        candidate = torch.tanh(input_new + reset * state_new)
        next_state = candidate + update * (hidden_state - candidate)
        return next_state @ self.output_weight + self.output_bias, next_state


def export_gru_int8(frame_model, behaviours, training_window_size, output_dir):
    model = copy.deepcopy(frame_model).cpu().float().eval()
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    int8_path = output_dir / "behaviour_gru_int8.onnx"
    batch = torch.export.Dim("batch")
    with TemporaryDirectory() as directory:
        fp32_path = Path(directory) / "model.onnx"
        torch.onnx.export(
            GRUFrameExport(model),
            (
                torch.zeros(2, model.cell.input_size),
                torch.zeros(2, model.cell.hidden_size),
            ),
            str(fp32_path),
            input_names=["embedding", "hidden_state"],
            output_names=["logits", "next_hidden_state"],
            dynamic_shapes=({0: batch}, {0: batch}),
            opset_version=18,
            dynamo=True,
            optimize=False,
            external_data=False,
        )
        graph = onnx.shape_inference.infer_shapes(onnx.load(fp32_path))
        onnx.save(graph, fp32_path)
        quantize_dynamic(
            str(fp32_path),
            str(int8_path),
            op_types_to_quantize=["MatMul"],
            weight_type=QuantType.QInt8,
            per_channel=True,
        )
    graph = onnx.load(int8_path)
    if sum(node.op_type == "MatMulInteger" for node in graph.graph.node) != 3:
        raise RuntimeError("Expected three INT8 weight projections")
    config = {
        "behaviours": list(behaviours),
        "hidden_channels": model.cell.hidden_size,
        "embedding_model_sha256": embedding_model_sha256(),
        "training_window_size": training_window_size,
    }
    onnx.helper.set_model_props(graph, {"behaviour_config": json.dumps(config)})
    onnx.checker.check_model(graph)
    onnx.save(graph, int8_path)
    return int8_path

In [ ]:
GRU_INT8_ONNX_PATH = export_gru_int8(
    models["GRU"].frame_model,
    BEHAVIOURS,
    WINDOW_SIZE,
    PROJECT_ROOT / "models_staging",
)
behaviour_session.cache_clear()

In [ ]:
# Evaluate quantised deployment inference
targets, predictions, elapsed = [], [], []
for track in val_ds.tracks:
    state, embedding = None, None
    for frame_hash in track.frame_hashes:
        if frame_hash is not None:
            embedding = track.embeddings[frame_hash]
        start = perf_counter_ns()
        result = classify_behaviour(embedding, state, GRU_INT8_ONNX_PATH)
        elapsed.append(perf_counter_ns() - start)
        state = result["next_hidden_state"]
        if frame_hash is not None:
            targets.append(track.behaviour_map[frame_hash])
            predictions.append(result["behaviour"])
print(classification_report(targets, predictions, labels=BEHAVIOURS, zero_division=0))
print(f"Median inference: {np.median(elapsed) / 1e6:.3f} ms")